In [2]:
import os
import random
import numpy as np
import cv2
from glob import glob
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [6]:
BASE_PATH = "/kaggle/input/datasets/aysendegerli/qatacov19-dataset/QaTa-COV19/QaTa-COV19-v2/Train Set"

IMAGE_DIR = os.path.join(BASE_PATH, "Images")
MASK_DIR = os.path.join(BASE_PATH, "Ground-truths")

image_paths = sorted(glob(os.path.join(IMAGE_DIR, "*.png")))
mask_paths = sorted(glob(os.path.join(MASK_DIR, "*.png")))

print("Total images:", len(image_paths))

Total images: 7145


In [7]:
class QaTaDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, resize=224):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.resize = resize

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.resize, self.resize))
        img = torch.tensor(img / 255.0, dtype=torch.float32).permute(2,0,1)

        if self.mask_paths is not None:
            mask = cv2.imread(self.mask_paths[idx], 0)
            mask = cv2.resize(mask, (self.resize, self.resize))
            mask = (mask > 0).astype(np.float32)
            mask = torch.tensor(mask).unsqueeze(0)
            return img, mask

        return img


In [8]:
combined = list(zip(image_paths, mask_paths))
random.shuffle(combined)
image_paths, mask_paths = zip(*combined)

num_clients = 4
client_data = [[] for _ in range(num_clients)]

for i, (img, msk) in enumerate(zip(image_paths, mask_paths)):
    client_data[i % num_clients].append((img, msk))

labeled_clients = [0, 1]
unlabeled_clients = [2, 3]

batch_size = 8
client_loaders = []

for i in range(num_clients):
    imgs = [x[0] for x in client_data[i]]
    masks = [x[1] for x in client_data[i]]
    dataset_client = QaTaDataset(imgs, masks)
    loader = DataLoader(dataset_client, batch_size=batch_size, shuffle=True)
    client_loaders.append(loader)

print("Clients ready")


Clients ready


In [20]:
class ResNetUNet(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        
        self.encoder0 = nn.Sequential(
            base_model.conv1,
            base_model.bn1,
            base_model.relu
        )
        self.pool = base_model.maxpool
        self.encoder1 = base_model.layer1
        self.encoder2 = base_model.layer2
        self.encoder3 = base_model.layer3
        self.encoder4 = base_model.layer4
        
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv4 = nn.Conv2d(256 + 256, 256, 3, padding=1)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = nn.Conv2d(128 + 128, 128, 3, padding=1)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = nn.Conv2d(64 + 64, 64, 3, padding=1)

        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.conv1 = nn.Conv2d(64 + 64, 64, 3, padding=1)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        input_size = x.shape[2:]

        e0 = self.encoder0(x)
        p = self.pool(e0)
        e1 = self.encoder1(p)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        d4 = self.up4(e4)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = F.relu(self.conv4(d4))

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = F.relu(self.conv3(d3))

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = F.relu(self.conv2(d2))

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e0], dim=1)
        d1 = F.relu(self.conv1(d1))

        out = self.final(d1)

        # Resize to original resolution (224x224)
        out = F.interpolate(
            out,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        return torch.sigmoid(out)


In [21]:
def dice_loss(pred, target, smooth=1e-6):
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return 1 - dice


In [22]:
def compute_entropy(pred):
    eps = 1e-6
    pred = torch.clamp(pred, eps, 1 - eps)
    entropy = - (pred * torch.log(pred) + (1 - pred) * torch.log(1 - pred))
    return entropy


In [23]:
def extract_prototypes(model, dataloader):
    model.eval()
    
    fg_sum = None
    bg_sum = None
    fg_count = 0
    bg_count = 0
    
    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            # Forward to encoder4
            e0 = model.encoder0(imgs)
            p = model.pool(e0)
            e1 = model.encoder1(p)
            e2 = model.encoder2(e1)
            e3 = model.encoder3(e2)
            features = model.encoder4(e3)  # [B,512,H,W]
            
            masks_resized = F.interpolate(
                masks,
                size=features.shape[2:],
                mode="nearest"
            )
            
            fg_mask = (masks_resized > 0).float()
            bg_mask = 1 - fg_mask
            
            fg_feat = features * fg_mask
            bg_feat = features * bg_mask
            
            if fg_sum is None:
                fg_sum = fg_feat.sum(dim=(0,2,3))
                bg_sum = bg_feat.sum(dim=(0,2,3))
            else:
                fg_sum += fg_feat.sum(dim=(0,2,3))
                bg_sum += bg_feat.sum(dim=(0,2,3))
            
            fg_count += fg_mask.sum()
            bg_count += bg_mask.sum()
    
    fg_proto = fg_sum / (fg_count + 1e-6)
    bg_proto = bg_sum / (bg_count + 1e-6)
    
    return fg_proto, bg_proto


In [24]:
def contrastive_loss(features, masks, fg_proto, bg_proto, temp=0.1):
    
    B, C, H, W = features.shape
    
    masks = F.interpolate(masks, size=(H, W), mode="nearest")
    
    features = features.permute(0,2,3,1).reshape(-1, C)
    masks = masks.reshape(-1)
    
    fg_proto = fg_proto.unsqueeze(0)
    bg_proto = bg_proto.unsqueeze(0)
    
    sim_fg = F.cosine_similarity(features, fg_proto, dim=1) / temp
    sim_bg = F.cosine_similarity(features, bg_proto, dim=1) / temp
    
    logits = torch.stack([sim_bg, sim_fg], dim=1)
    
    labels = masks.long()
    
    loss = F.cross_entropy(logits, labels)
    
    return loss


In [25]:
def compute_similarity_weights(unlabeled_proto, labeled_protos):
    sims = []
    for proto in labeled_protos:
        sim = F.cosine_similarity(
            unlabeled_proto.unsqueeze(0),
            proto.unsqueeze(0),
            dim=1
        )
        sims.append(sim)
    
    sims = torch.stack(sims).squeeze()
    weights = torch.softmax(sims, dim=0)
    return weights


In [26]:
def compute_agreement_ratio(local_model, agg_model, dataloader, threshold=0.5):
    local_model.eval()
    agg_model.eval()
    
    total = 0
    agree = 0
    
    with torch.no_grad():
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            
            p1 = (local_model(imgs) > threshold).float()
            p2 = (agg_model(imgs) > threshold).float()
            
            agreement = (p1 == p2).float()
            
            agree += agreement.sum().item()
            total += agreement.numel()
    
    return agree / (total + 1e-6)


In [27]:
def local_train_full(model,
                     dataloader,
                     client_id,
                     labeled_agg_model=None,
                     fg_proto=None,
                     bg_proto=None,
                     epochs=1,
                     lr=1e-4,
                     threshold=0.5,
                     lambda_contrast=0.1):
    
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        total_loss = 0
        
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            # Forward full model
            preds = model(imgs)
            dice = dice_loss(preds, masks)
            
            # Extract encoder features for contrastive
            e0 = model.encoder0(imgs)
            p = model.pool(e0)
            e1 = model.encoder1(p)
            e2 = model.encoder2(e1)
            e3 = model.encoder3(e2)
            features = model.encoder4(e3)
            
            contrast = contrastive_loss(
                features,
                masks,
                fg_proto,
                bg_proto
            )
            
            loss = dice + lambda_contrast * contrast
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Client {client_id} Epoch {epoch+1} Loss: {total_loss/len(dataloader):.4f}")
    
    return model.state_dict()


In [28]:
def build_labeled_agg_model(global_model, labeled_models, weights):
    
    new_model = ResNetUNet().to(device)
    new_state = new_model.state_dict()
    
    labeled_states = [m.state_dict() for m in labeled_models]
    
    for key in new_state.keys():
        weighted_sum = 0
        for i in range(len(labeled_states)):
            weighted_sum += weights[i] * labeled_states[i][key]
        new_state[key] = weighted_sum
    
    new_model.load_state_dict(new_state)
    return new_model


In [29]:
global_model = ResNetUNet().to(device)

rounds = 15
local_epochs = 1

for r in range(rounds):
    print(f"\n================ ROUND {r+1} ================")
    
    client_weights = [None]*num_clients
    client_models = {}
    labeled_models = []
    labeled_fg_protos = []
    labeled_bg_protos = []
    
    # -------- TRAIN LABELED CLIENTS --------
    for i in labeled_clients:
        local_model = ResNetUNet().to(device)
        local_model.load_state_dict(global_model.state_dict())
        
        # Extract prototypes from labeled client
        fg_proto, bg_proto = extract_prototypes(local_model, client_loaders[i])
        
        weights = local_train_full(
            local_model,
            client_loaders[i],
            i,
            fg_proto=fg_proto,
            bg_proto=bg_proto,
            epochs=local_epochs
        )
        
        client_weights[i] = weights
        client_models[i] = local_model
        
        labeled_models.append(local_model)
        labeled_fg_protos.append(fg_proto)
        labeled_bg_protos.append(bg_proto)
    
    # -------- UNLABELED CLIENTS --------
    for i in unlabeled_clients:
        temp_model = ResNetUNet().to(device)
        temp_model.load_state_dict(global_model.state_dict())
        
        # Use average labeled prototypes
        fg_proto = torch.stack(labeled_fg_protos).mean(dim=0)
        bg_proto = torch.stack(labeled_bg_protos).mean(dim=0)
        
        weights = local_train_full(
            temp_model,
            client_loaders[i],
            i,
            fg_proto=fg_proto,
            bg_proto=bg_proto,
            epochs=local_epochs
        )
        
        client_weights[i] = weights
        client_models[i] = temp_model
    
    # -------- DYNAMIC AGGREGATION --------
    Z = []
    data_sizes = []
    
    for i in range(num_clients):
        data_sizes.append(len(client_loaders[i].dataset))
        
        if i in labeled_clients:
            Z.append(1.0)
        else:
            Z.append(0.7)  # simplified stability factor
    
    Z = torch.tensor(Z)
    data_sizes = torch.tensor(data_sizes, dtype=torch.float32)
    
    weights_dyn = Z * data_sizes
    weights_dyn = weights_dyn / weights_dyn.sum()
    
    print("Dynamic weights:", weights_dyn.numpy())
    
    global_dict = global_model.state_dict()
    
    for key in global_dict.keys():
        weighted_sum = 0
        for i in range(num_clients):
            weighted_sum += weights_dyn[i] * client_weights[i][key]
        global_dict[key] = weighted_sum
    
    global_model.load_state_dict(global_dict)



================ ROUND 1 ================
Client 0 Epoch 1 Loss: 0.4226
Client 1 Epoch 1 Loss: 0.4295
Client 2 Epoch 1 Loss: 0.4178
Client 3 Epoch 1 Loss: 0.4254
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 2 ================
Client 0 Epoch 1 Loss: 0.2329
Client 1 Epoch 1 Loss: 0.2343
Client 2 Epoch 1 Loss: 0.2268
Client 3 Epoch 1 Loss: 0.2289
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 3 ================
Client 0 Epoch 1 Loss: 0.2063
Client 1 Epoch 1 Loss: 0.2089
Client 2 Epoch 1 Loss: 0.1988
Client 3 Epoch 1 Loss: 0.2066
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 4 ================
Client 0 Epoch 1 Loss: 0.1963
Client 1 Epoch 1 Loss: 0.1937
Client 2 Epoch 1 Loss: 0.1888
Client 3 Epoch 1 Loss: 0.1981
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 5 ================
Client 0 Epoch 1 Loss: 0.1773
Client 1 Epoch 1 Loss: 

In [30]:
# Validation Split
val_imgs = image_paths[:200]
val_masks = mask_paths[:200]

val_dataset = QaTaDataset(val_imgs, val_masks)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


def evaluate(model, dataloader):
    model.eval()
    total_dice = 0
    count = 0
    
    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            preds = model(imgs)
            preds = (preds > 0.5).float()
            
            intersection = (preds * masks).sum(dim=(1,2,3))
            union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
            
            dice = (2 * intersection + 1e-6) / (union + 1e-6)
            
            total_dice += dice.sum().item()
            count += imgs.size(0)
    
    return total_dice / count


dice_score = evaluate(global_model, val_loader)

print("\n================ FINAL FSSS RESULT ================")
print("FSSS Validation Dice:", dice_score)



================ FINAL FSSS RESULT ================
FSSS Validation Dice: 0.8421880745887756


In [31]:
baseline_model = ResNetUNet().to(device)

rounds = 15
local_epochs = 1

for r in range(rounds):
    print(f"\n===== BASELINE ROUND {r+1} =====")
    
    client_weights = []
    
    for i in labeled_clients:  # Only labeled clients
        local_model = ResNetUNet().to(device)
        local_model.load_state_dict(baseline_model.state_dict())
        
        optimizer = torch.optim.Adam(local_model.parameters(), lr=1e-4)
        local_model.train()
        
        for epoch in range(local_epochs):
            total_loss = 0
            for imgs, masks in client_loaders[i]:
                imgs = imgs.to(device)
                masks = masks.to(device)
                
                preds = local_model(imgs)
                loss = dice_loss(preds, masks)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            print(f"Client {i} Epoch Loss: {total_loss/len(client_loaders[i]):.4f}")
        
        client_weights.append(local_model.state_dict())
    
    # FedAvg between labeled clients
    global_dict = baseline_model.state_dict()
    
    for key in global_dict.keys():
        stacked = torch.stack(
            [client_weights[i][key].float() for i in range(len(client_weights))],
            dim=0
        )
        global_dict[key] = stacked.mean(dim=0)
    
    baseline_model.load_state_dict(global_dict)


# Evaluate baseline
dice_baseline = evaluate(baseline_model, val_loader)

print("\n================ BASELINE RESULT ================")
print("Baseline Dice:", dice_baseline)



===== BASELINE ROUND 1 =====
Client 0 Epoch Loss: 0.3882
Client 1 Epoch Loss: 0.3979

===== BASELINE ROUND 2 =====
Client 0 Epoch Loss: 0.2193
Client 1 Epoch Loss: 0.2218

===== BASELINE ROUND 3 =====
Client 0 Epoch Loss: 0.1919
Client 1 Epoch Loss: 0.2006

===== BASELINE ROUND 4 =====
Client 0 Epoch Loss: 0.1738
Client 1 Epoch Loss: 0.1786

===== BASELINE ROUND 5 =====
Client 0 Epoch Loss: 0.1715
Client 1 Epoch Loss: 0.1720

===== BASELINE ROUND 6 =====
Client 0 Epoch Loss: 0.1529
Client 1 Epoch Loss: 0.1596

===== BASELINE ROUND 7 =====
Client 0 Epoch Loss: 0.1519
Client 1 Epoch Loss: 0.1555

===== BASELINE ROUND 8 =====
Client 0 Epoch Loss: 0.1214
Client 1 Epoch Loss: 0.1224

===== BASELINE ROUND 12 =====
Client 0 Epoch Loss: 0.1145
Client 1 Epoch Loss: 0.1186

===== BASELINE ROUND 13 =====
Client 0 Epoch Loss: 0.1091
Client 1 Epoch Loss: 0.1164

===== BASELINE ROUND 14 =====
Client 0 Epoch Loss: 0.1066
Client 1 Epoch Loss: 0.1065

===== BASELINE ROUND 15 =====
Client 0 Epoch Loss:

### Save model for the FEDSEG web app

Run the cell below to save the trained `global_model` so that the Streamlit app (`model/infer.py`) can load it. Save from the **project root** (e.g. where `frontend/` and `model/` live), or adjust the path.

In [ ]:
# Save global_model for the web app (run from project root or set path to model/fedseg_model.pt)
import os
save_path = "model/fedseg_model.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(global_model.state_dict(), save_path)
print(f"Saved to {save_path}")